# CrediScope Analysis

**Clear decisions. Clear reasons.**

This notebook explores the CreditSense dataset and prepares the data for machine learning.

The aim is to understand the structure of the dataset, identify any issues that need preprocessing, and prepare the features before training a credit-risk model.

In [4]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv("../data/creditsense_applications.csv")
dictionary = pd.read_csv("../data/creditsense_data_dictionary.csv")

In [6]:
df.shape

(6000, 27)

In [7]:
df.head()

,application_id,age_years,sex,marital_status,residency_status,num_dependents,region,employment_status,months_in_current_job,annual_income_gbp,...,has_savings_account,savings_balance_gbp,checking_account_status,loan_amount_gbp,loan_term_months,loan_purpose,has_guarantor,previous_application_declined,application_channel,default_within_24m
0,APP-000001,34,Female,Married,Settled Status,1,North West,Permanent,19,66550,...,Yes,380.0,500 GBP or more,13100,36,Home improvement,No,No,Online,0
1,APP-000002,43,Female,Married,Settled Status,1,South East,Permanent,29,20820,...,No,0.0,Overdrawn,9700,60,Medical,No,No,Branch,1
2,APP-000003,47,Male,Single,UK Citizen,0,South East,Permanent,115,32710,...,No,0.0,500 GBP or more,8600,36,Home improvement,No,No,Online,0
3,APP-000004,63,Male,Married,UK Citizen,1,London,Permanent,146,47300,...,Yes,NaN,0 to 500 GBP,25600,36,Home improvement,No,No,Online,0
4,APP-000005,40,Male,Single,UK Citizen,1,Yorkshire,Permanent,22,41400,...,Yes,860.0,0 to 500 GBP,3500,36,Car,Yes,No,Branch,0


In [8]:
df.columns.tolist()

['application_id',
 'age_years',
 'sex',
 'marital_status',
 'residency_status',
 'num_dependents',
 'region',
 'employment_status',
 'months_in_current_job',
 'annual_income_gbp',
 'monthly_expenses_gbp',
 'housing_status',
 'years_at_current_address',
 'existing_debt_gbp',
 'num_existing_credit_lines',
 'num_late_payments_24m',
 'credit_history',
 'has_savings_account',
 'savings_balance_gbp',
 'checking_account_status',
 'loan_amount_gbp',
 'loan_term_months',
 'loan_purpose',
 'has_guarantor',
 'previous_application_declined',
 'application_channel',
 'default_within_24m']

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   application_id                 6000 non-null   str    
 1   age_years                      6000 non-null   int64  
 2   sex                            6000 non-null   str    
 3   marital_status                 6000 non-null   str    
 4   residency_status               6000 non-null   str    
 5   num_dependents                 6000 non-null   int64  
 6   region                         6000 non-null   str    
 7   employment_status              6000 non-null   str    
 8   months_in_current_job          6000 non-null   int64  
 9   annual_income_gbp              6000 non-null   int64  
 10  monthly_expenses_gbp           6000 non-null   int64  
 11  housing_status                 6000 non-null   str    
 12  years_at_current_address       5832 non-null   float64
 13 

In [10]:
df.isna().sum().sort_values(ascending=False)

savings_balance_gbp              258
years_at_current_address         168
application_id                     0
marital_status                     0
residency_status                   0
num_dependents                     0
region                             0
employment_status                  0
months_in_current_job              0
age_years                          0
sex                                0
monthly_expenses_gbp               0
annual_income_gbp                  0
existing_debt_gbp                  0
housing_status                     0
num_existing_credit_lines          0
num_late_payments_24m              0
credit_history                     0
has_savings_account                0
checking_account_status            0
loan_amount_gbp                    0
loan_term_months                   0
loan_purpose                       0
has_guarantor                      0
previous_application_declined      0
application_channel                0
default_within_24m                 0
d

## Missing Values

Only two features contain missing values:

- `savings_balance_gbp` has 258 missing values.
- `years_at_current_address` has 168 missing values.

All other columns are complete.

These missing values will be investigated before deciding how they should be handled during preprocessing.

In [11]:
df["default_within_24m"].value_counts()

default_within_24m
0    4467
1    1533
Name: count, dtype: int64

In [12]:
df["default_within_24m"].value_counts(normalize=True) * 100

default_within_24m
0    74.45
1    25.55
Name: proportion, dtype: float64

## Target Variable

The target variable is `default_within_24m`.

- `0` means the applicant did not default within 24 months.
- `1` means the applicant did default within 24 months.

The target is moderately imbalanced, with 25.55% of applicants in the default class.

In [13]:
dictionary.head(30)

,column,type,description,protected_attribute,notes
0,application_id,string,Unique application reference,No,Identifier only - must NOT be used as a feature
1,age_years,integer,Applicant age at application,Yes,Protected characteristic under Equality Act 2010
2,sex,categorical,Female / Male,Yes,Protected characteristic under Equality Act 2010
3,marital_status,categorical,Single / Married / Divorced / Widowed,Yes,Correlates strongly with age
4,residency_status,categorical,UK Citizen / Settled Status / Visa Holder,Yes,Proxy for nationality - protected characteristic
5,num_dependents,integer,Number of financial dependents,No,Correlates with marital_status
6,region,categorical,UK region of residence,No,Can act as a proxy for other attributes
7,employment_status,categorical,Permanent / Fixed-term / Self-employed / Zero-...,No,Retired appears only in older applicants
8,months_in_current_job,integer,Tenure in current employment (months),No,Zero for unemployed applicants - structural ze...
9,annual_income_gbp,integer,Gross annual income in GBP,No,Not adjusted for region


## Initial Data Dictionary Observations

The data dictionary highlights several features that will need careful handling during preprocessing.

- `application_id` is an identifier and should not be used as a predictive feature.
- `age_years`, `sex`, `marital_status`, `residency_status` and `housing_status` are marked as protected attributes.
- `region` is not marked as protected, but it may act as a proxy for other characteristics.
- `credit_history` is ordinal.
- `checking_account_status` is ordinal, but the `No account` category does not fit naturally on the same ordered scale.
- `months_in_current_job = 0` for unemployed applicants is a structural zero and represents valid information.
- Missing values occur in `years_at_current_address` and `savings_balance_gbp`.

In [14]:
numerical_columns = df.select_dtypes(
   include=["int64", "float64"] 
).columns.tolist()

categorical_columns = df.select_dtypes(
   include="str" 
).columns.tolist()

In [15]:
print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)


Numerical Columns:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months', 'default_within_24m']

Categorical Columns:
['application_id', 'sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


In [16]:
target_column = "default_within_24m"
id_column = "application_id"

numerical_features = [
    col for col in numerical_columns 
    if col not in [target_column, id_column]
]

categorical_features = [
    col for col in categorical_columns
    if col != id_column
]

In [17]:
print("Numerical Columns:") 
print(numerical_features)

print("\nCategorical Columns:")
print(categorical_features)

Numerical Columns:
['age_years', 'num_dependents', 'months_in_current_job', 'annual_income_gbp', 'monthly_expenses_gbp', 'years_at_current_address', 'existing_debt_gbp', 'num_existing_credit_lines', 'num_late_payments_24m', 'savings_balance_gbp', 'loan_amount_gbp', 'loan_term_months']

Categorical Columns:
['sex', 'marital_status', 'residency_status', 'region', 'employment_status', 'housing_status', 'credit_history', 'has_savings_account', 'checking_account_status', 'loan_purpose', 'has_guarantor', 'previous_application_declined', 'application_channel']


## Feature Groups

The dataset contains 25 predictive features.

- 12 numerical features
- 13 categorical features
- `application_id` is excluded because it is only an identifier
- `default_within_24m` is the target variable

# Data Preprocessing

This section prepares the dataset for machine learning.

A separate working copy of the original dataframe will be used so that the raw data remains unchanged while preprocessing decisions are applied.

In [18]:
df_clean = df.copy()

In [19]:
print("Original dataset shape:", df.shape)
print("Working copy shape:", df_clean.shape)

Original dataset shape: (6000, 27)
Working copy shape: (6000, 27)


In [20]:
df_clean.isna().sum()[df_clean.isna().sum() >0]

years_at_current_address    168
savings_balance_gbp         258
dtype: int64

## Missing Value Investigation

Two features contain missing values:

- `years_at_current_address`
- `savings_balance_gbp`

Before filling these values, their distributions and meanings will be checked to choose an appropriate preprocessing method.

In [21]:
df_clean["years_at_current_address"].describe()

count    5832.000000
mean        5.169513
std         4.893061
min         0.000000
25%         1.900000
50%         3.700000
75%         6.800000
max        40.000000
Name: years_at_current_address, dtype: float64

In [22]:
df_clean["savings_balance_gbp"].describe()

count      5742.000000
mean       3206.849530
std        7875.004884
min           0.000000
25%           0.000000
50%         610.000000
75%        3250.000000
max      160000.000000
Name: savings_balance_gbp, dtype: float64

In [25]:
df_clean.groupby("has_savings_account")["savings_balance_gbp"].apply(
    lambda x: x.isna().sum()
)

has_savings_account
No       0
Yes    258
Name: savings_balance_gbp, dtype: int64

## Missing Value Decision

The two features with missing values are `years_at_current_address` and `savings_balance_gbp`.

Both features are positively skewed, so the median is preferred to the mean because it is less affected by large values.

For `savings_balance_gbp`, all missing values occur among applicants who do have a savings account. Applicants without a savings account already have a meaningful balance of £0, so those zeros must be preserved.

In [28]:
df_clean["years_at_current_address"] = (
    df_clean["years_at_current_address"]
    .fillna(df_clean["years_at_current_address"].median())
)

df_clean["savings_balance_gbp"] = (
    df_clean["savings_balance_gbp"]
    .fillna(df_clean["savings_balance_gbp"].median())
)

In [29]:
df_clean[
    ["years_at_current_address", "savings_balance_gbp"]
].isna().sum()

years_at_current_address    0
savings_balance_gbp         0
dtype: int64

In [30]:
df_clean.isna().sum().sum()

np.int64(0)

### Missing Value Result

Median imputation was applied to the two features containing missing values.

After preprocessing, the working dataset contains no missing values.

In [32]:
df_clean.groupby("employment_status")["months_in_current_job"].describe()

,count,mean,std,min,25%,50%,75%,max
employment_status,,,,,,,,
Fixed-term,606.0,21.016502,13.643140,2.0,11.0,17.0,29.0,89.0
Permanent,4125.0,53.890667,34.846809,3.0,28.0,46.0,73.0,245.0
Retired,77.0,111.389610,44.850382,41.0,74.0,109.0,136.0,242.0
Self-employed,679.0,55.587629,37.494002,3.0,30.0,46.0,72.0,250.0
Unemployed,227.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Zero-hours,286.0,21.363636,13.552003,1.0,11.0,19.0,30.0,83.0


## Structural Zero Check

The `months_in_current_job` feature contains zero values for unemployed applicants.

All 227 unemployed applicants have `months_in_current_job = 0`, which indicates that these values are structural zeros rather than missing data.

These values will therefore be preserved.

In [33]:
binary_columns = [
    "has_savings_account",
    "has_guarantor",
    "previous_application_declined",
]

In [34]:
for col in binary_columns:
    df_clean[col] = df_clean[col].map({"Yes": 1, "No": 0})

In [35]:
for col in binary_columns:
    print(col, df_clean[col].value_counts())

has_savings_account has_savings_account
1    3495
0    2505
Name: count, dtype: int64
has_guarantor has_guarantor
0    5432
1     568
Name: count, dtype: int64
previous_application_declined previous_application_declined
0    5251
1     749
Name: count, dtype: int64


## Binary Feature Encoding

Three Yes/No categorical features were converted to numerical binary values:

- `has_savings_account`
- `has_guarantor`
- `previous_application_declined`

`Yes` was encoded as `1` and `No` as `0`.